In [1]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
#!pip install psutil

In [2]:
def pdf_loader(data):
    loader=PyPDFDirectoryLoader(data)
    documents=loader.load()
    return documents

data=pdf_loader(r"C:\Users\shukl\Desktop\Maharashtra_Textbook_State_Board_Bot\Maharashtra_tb_6th_to_12th_books_chat_bot_RAG\data\textbooks")

In [3]:
data

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2022-07-04T16:27:16+05:30', 'moddate': '2022-07-04T16:27:16+05:30', 'title': '', 'source': 'C:\\Users\\shukl\\Desktop\\Maharashtra_Textbook_State_Board_Bot\\Maharashtra_tb_6th_to_12th_books_chat_bot_RAG\\data\\textbooks\\10th_english.pdf', 'total_pages': 204, 'page': 0, 'page_label': '1'}, page_content='B§J«Or Hw$‘ma^maVr B¶ËVm Xhmdr (B§J«Or ‘mÜ¶‘) 73.00\nMAHARASHTRA STATE BUREAU OF TEXTBOOK PRODUCTION AND CURRICULUM RESEARCH, PUNE.\nKUMARBHARATI\nSTANDARD TEN\nENGLISH\nB§J«Or Hw$‘ma^maVr B¶ËVm Xhmdr (B§J«Or ‘mÜ¶‘)'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2022-07-04T16:27:16+05:30', 'moddate': '2022-07-04T16:27:16+05:30', 'title': '', 'source': 'C:\\Users\\shukl\\Desktop\\Maharashtra_Textbook_State_Board_Bot\\Maharashtra_tb_6th_to_12th_books_chat_bot_RAG\\data\\textbooks\\10th_englis

In [105]:
len(data)

1262

In [4]:
# To get particular needed content from the data

from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [5]:
minimal_data=filter_to_minimal_docs(data)

In [6]:
# Text Splitting into chunks
def text_split(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(chunk_overlap=30,chunk_size=500)
    
    text_chunks=text_splitter.split_documents(minimal_data)
    return text_chunks

In [7]:
text_chunk=text_split(minimal_data)

In [8]:
len(text_chunk)

29622

In [10]:
# Embeddings 
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [12]:
vector=embeddings.embed_query("Hi")
vector

[-0.09047624468803406,
 0.04043957591056824,
 0.023905588313937187,
 0.058948010206222534,
 -0.022882280871272087,
 -0.047220077365636826,
 0.0450475811958313,
 0.015786297619342804,
 -0.04819947108626366,
 -0.03779413923621178,
 -0.019077599048614502,
 0.02130882628262043,
 -0.004683048464357853,
 -0.04330812022089958,
 0.05991485342383385,
 0.059103354811668396,
 -0.02803678624331951,
 -0.05921844020485878,
 -0.12440310418605804,
 -0.03560001030564308,
 -0.006080544088035822,
 0.0324290506541729,
 -0.037800729274749756,
 0.02471098117530346,
 -0.04272434860467911,
 -0.04245390743017197,
 0.04593564197421074,
 0.0986255630850792,
 -0.04999805614352226,
 -0.03523589298129082,
 0.07083974778652191,
 0.03316323459148407,
 0.02658829838037491,
 0.0001732416421873495,
 0.0038816838059574366,
 0.030467195436358452,
 -0.07820260524749756,
 -0.12037956714630127,
 0.01804153434932232,
 0.02282906509935856,
 -0.0017749639227986336,
 -0.023449871689081192,
 0.003058134810999036,
 0.0243557728826

In [ ]:
print("Vector length",len(vector))# 384 is the dimension of the vector for the given model

Vector length 384


: 

In [13]:
# Using pinecone for vector database

## Pinecone Connection

In [14]:
from dotenv import load_dotenv
load_dotenv()
import os

pinecone_api_key=os.getenv("PINECONE_API_KEY")
groq_api_key=os.getenv("GROQ_API_KEY")

In [15]:
from pinecone import Pinecone
pinecone = Pinecone(api_key=pinecone_api_key)
pinecone

In [16]:
# Going to create database in pinecone and then will add the vectors to it.
from pinecone import ServerlessSpec

index_name="textbookbot"
if not pinecone.has_index(index_name):
    pinecone.create_index(name=index_name,dimension=384,metric="cosine",spec=ServerlessSpec(cloud="aws",region="us-east-1"))

index=pinecone.Index(index_name)

In [17]:
index.describe_index_stats()

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [18]:
# Now try to store the vector in the index

from langchain_pinecone import PineconeVectorStore
vectorstore=PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embeddings,
    index_name=index_name
    
)

In [19]:
# Load the existing index

# “Load existing index” ka matlab kya?
# 👉 Pinecone me jo index pehle se bana hua hai + data store hai

vectorstore=PineconeVectorStore.from_existing_index(
    embedding=embeddings,
    index_name=index_name,
)

In [20]:
docs=vectorstore.similarity_search("Whats there in the 8th class English textbook?")

In [21]:
docs

[Document(id='6148328a-d469-439c-be29-472ae9cabf83', metadata={'source': 'C:\\Users\\shukl\\Desktop\\Maharashtra_Textbook_State_Board_Bot\\Maharashtra_tb_6th_to_12th_books_chat_bot_RAG\\data\\textbooks\\8th_history.pdf'}, page_content='standard VIII in your hands. \nThe structure of the textbook is designed with the objective that the subject \nshould be properly understood, felt interesting and get inspired by the work done \nby our ancestors. By studying this textbook we hope that along with knowledge \nyour learning will also become meaningful. For this purpose coloured pictures, \nmaps are given in the textbook. Every chapter of the textbook should be studied'),
 Document(id='3e16db8b-2d9f-4dde-a906-9e6d7d554e7f', metadata={'source': 'C:\\Users\\shukl\\Desktop\\Maharashtra_Textbook_State_Board_Bot\\Maharashtra_tb_6th_to_12th_books_chat_bot_RAG\\data\\textbooks\\11th_chemistry.pdf'}, page_content='concepts mentioned in chapter.'),
 Document(id='b3f3c1f6-100d-4d21-87c7-415f0d804e58',

In [22]:
for doc in docs:
    print(doc.page_content)
    print("=" * 50)

standard VIII in your hands. 
The structure of the textbook is designed with the objective that the subject 
should be properly understood, felt interesting and get inspired by the work done 
by our ancestors. By studying this textbook we hope that along with knowledge 
your learning will also become meaningful. For this purpose coloured pictures, 
maps are given in the textbook. Every chapter of the textbook should be studied
concepts mentioned in chapter.
textbook. Many of the activities are designed to show you ways of thinking and 
learning on your own. The more you use them, the better you will learn.
We have focussed upon linguistic items in the Language Study (Grammar 
and Vocubulary) activities. The textbook also aims to help students to attain a 
proficiency level in English, whereby you can directly ‘think’ in English rather 
than think in your mother tounge and translate your thoughts into English. This
The efforts taken to prepare the textbook will not only enrich the learn

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 15
    }
)

In [ ]:
retriever.invoke("What is lump in knee?")

[Document(id='ef4011ef-aa51-4930-ab47-a6d0e579f4b3', metadata={'source': 'C:\\Users\\shukl\\Desktop\\Maharashtra_Textbook_State_Board_Bot\\Maharashtra_tb_6th_to_12th_books_chat_bot_RAG\\data\\textbooks\\8th_science.pdf'}, page_content='with our finger a hard bump that seems to move. \nThis is nothing but larynx. As shown in Figure \n15.5, it is at the upper end of the windpipe. Two \nvocal cords, are stretched across the voice box or \nlarynx in such a way that it leaves a narrow slit \nbetween them for the passage of air.  \nWhen the lungs force air through the slit, the \nvocal cords vibrate, producing sound. Muscles \nattached to the vocal cords can make the cords \ntight or loose. When the vocal cords are tight and'),
 Document(id='53a379ed-59a5-4cfb-a0b3-35b8e2abb6da', metadata={'source': 'C:\\Users\\shukl\\Desktop\\Maharashtra_Textbook_State_Board_Bot\\Maharashtra_tb_6th_to_12th_books_chat_bot_RAG\\data\\textbooks\\10th_science_part2.pdf'}, page_content='Cyst \nBulge Daughter \ny

In [ ]:
docs = retriever.invoke("sound production")

In [ ]:
for doc in docs:
    print(doc.page_content)

• Field Sound Engine er etc.
There are various audio editing 
3
a)  Radio Techn ician
  b)  Field Sound Engineer
  c)  Film Sound Recording
  d)  Choreography
 3. Which of the following is not a 
audio editing software?
  a)  Traverso
  b)  Mixxx
  c)  Adobe Audio Editor
  d)  Ardour
 4.  Select the most approriate option 
from the following.
  a)  Audio edit ing is a process 
where we record the audio 
first and then make it suitable 
for listening.
  b)  Audio editin g deals with 
recording audio so that it will 
be easy for editing.
107
‘Apps’ for generation of different sound notes (sound note 
generator app) may be available on cell-phones. With the help of your 
teacher, using such an app, generate sound notes listed in the table.
Sound Produced by Human
Either speak a little loudly or sing a song 
or produce humming sound like a bee and put 
your fingers on your throat. Do you feel some 
vibrations?
In the humans, sound is produced in the  
larynx. While swallowing food, we can 

In [ ]:
# Creating LLM model
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.3-70b-versatile",temperature=0.7)

In [ ]:
model.invoke("What is sound?")

AIMessage(content='Sound is a form of vibration that travels through a medium, such as air, water, or solids, and is perceived by the ear and interpreted by the brain. It is a type of mechanical wave that is caused by the back-and-forth motion of particles in the medium.\n\nWhen an object vibrates, it creates a disturbance in the surrounding medium, which then transfers the energy of the vibration to neighboring particles. This energy transfer creates a series of compressions and rarefactions (expansions) in the medium, which propagate outward from the source of the vibration.\n\nThe characteristics of sound waves include:\n\n1. **Frequency**: The number of oscillations or cycles per second, measured in Hertz (Hz). Frequency determines the pitch of the sound.\n2. **Amplitude**: The magnitude of the vibration, which determines the loudness of the sound.\n3. **Wavelength**: The distance between two consecutive compressions or rarefactions, which is related to the frequency and speed of t

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

: 

In [ ]:
system_prompt = (
    "You are an expert educational assistant for Maharashtra State Board "
    "(SSC and HSC) textbooks from classes 6th to 12th. "
    
    "Answer questions only from the provided textbook context. "
    "Use the retrieved context carefully and give accurate, student-friendly answers. "
    
    "You are especially skilled in:\n"
    "- Physics\n"
    "- Chemistry\n"
    "- Biology\n"
    "- Mathematics\n"
    "- Science\n"
    "- PCMB subjects\n"
    
    "Rules:\n"
    "1. Do NOT make up answers.\n"
    "2. If the answer is not present in the context, clearly say:\n"
    "'I could not find the answer in the provided Maharashtra State Board textbook context.'\n"
    "3. For Mathematics:\n"
    "- Solve step-by-step whenever possible.\n"
    "- Show formulas used.\n"
    "- Keep calculations clear and simple.\n"
    "4. For Physics and Chemistry:\n"
    "- Explain definitions, laws, formulas, derivations, and concepts clearly.\n"
    "- Mention units where needed.\n"
    "5. For Biology:\n"
    "- Explain processes and diagrams in simple language.\n"
    "6. Keep answers concise but complete.\n"
    "7. Prefer textbook terminology and textbook-style explanations.\n"
    "8. If multiple textbook chunks are provided, combine them intelligently.\n"
    
    "\nRetrieved textbook context:\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

: 

In [ ]:
question_answer_chain = create_stuff_documents_chain(model, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

: 

In [ ]:
response = rag_chain.invoke({"input": "What is Sound"})
print(response["answer"])

response

Sound is a form of energy that is produced by vibrations. When an object vibrates, it creates a disturbance in the air particles around it, causing them to oscillate back and forth. This oscillation, or wave motion, is what we perceive as sound.

In the context of the provided textbook, sound is generated by the vibration of objects, such as the prongs of a tuning fork or the screen of a loudspeaker. These vibrations create compressions and rarefactions in the air, which are propagated away from the source and can be heard by our ears.

As mentioned in the textbook, "If sound waves are generated in air, what moves away from the source? Is it the air itself or the state of compression and rarefaction created in the air?" The answer is that it is the state of compression and rarefaction created in the air that moves away from the source, not the air itself. This is what we perceive as sound.


{'input': 'What is Sound',
 'context': [Document(id='36d6e2bb-3afd-427b-94cc-1f3d735a07a6', metadata={'source': 'C:\\Users\\shukl\\Desktop\\Maharashtra_Textbook_State_Board_Bot\\Maharashtra_tb_6th_to_12th_books_chat_bot_RAG\\data\\textbooks\\8th_science.pdf'}, page_content='of hearing a sound. \nIf sound waves are generated in air, what moves away \nfrom the source? Is it the air itself or the state of compression \nand rarefaction created in the air? \nCan you recall?\nUse your brain power'),
  Document(id='4e7cb9fb-f46a-4e47-b347-98e8c680937f', metadata={'source': 'C:\\Users\\shukl\\Desktop\\Maharashtra_Textbook_State_Board_Bot\\Maharashtra_tb_6th_to_12th_books_chat_bot_RAG\\data\\textbooks\\8th_science.pdf'}, page_content='15.2 (b), region A) transfer their energy to the air molecules in the next region (region B). So, \nthe air in that region goes to compressed state (See Figure 15.2 (c), region B). Such a periodic \nmotion of the prongs creates compression and rarefaction in the a

: 

In [ ]:
response = rag_chain.invoke({"input": "what is Acne?"})
print(response["answer"])

I could not find the answer in the provided Maharashtra State Board textbook context.


: 

In [ ]:
docs = retriever.invoke(
    "List of poems in 10th english"
)

for doc in docs:
    print(doc.metadata)
    print(doc.page_content[:1000])
    print("="*100)

{'source': 'C:\\Users\\shukl\\Desktop\\Maharashtra_Textbook_State_Board_Bot\\Maharashtra_tb_6th_to_12th_books_chat_bot_RAG\\data\\textbooks\\10th_english.pdf'}
6
l Favourite line
l Theme/Central idea
l Figures of speech
	l Special features - Type of the poem, language, tone, implied meaning, etc.
l Why I like/ dislike the poem
9. Imagine that you have to deliver a speech on the occasion of ‘Independence
Day’ or the ‘Republic Day’ in the school assembly. Prepare a speech to deliver
on ‘India of my dreams’
Use the following steps :
l Greeting and salutation
l Self Introduction
l Introduction of the topic
l Elaboration of the topic with examples
{'source': 'C:\\Users\\shukl\\Desktop\\Maharashtra_Textbook_State_Board_Bot\\Maharashtra_tb_6th_to_12th_books_chat_bot_RAG\\data\\textbooks\\10th_english.pdf'}
(1)  fear
(2) 
9. Write an appreciation of the poem in a paragraph format. 
 (Refer to page no. 5.) 
10. Project
 Prepare a Presentation (on paper or on a PC) as a piece of reference to oth

: 

: 

: 

: 

: 